## Model predictions
Import libraries

In [1]:
import os
import utils
import numpy as np
import pandas as pd
import tensorflow as tf

from silx.io.dictdump import h5todict

Define inputs

In [2]:
model_name = "meno"
# Data indices
indices = {
  "x": [list(range(4)),[4],[5],[6]],
  "y": [list(range(7,11)), [11]]
}
# Paths
path_to_data = "./../datagen/data/test/"
path_to_model = "./../03.merged/case.r06/saving/plasma/SavedModel/"
path_to_init_sols = "./../datagen/datagen/samples/init_sols.hdf5"
path_to_saving = "./output/visual/"
os.makedirs(path_to_saving, exist_ok=True)
# Plotting
labels = {
  "Ar": r"Ar",
  "Ar4s": r"Ar$^{4s}$",
  "Ar4p": r"Ar$^{4p}$",
  "Ar+": r"Ar$^+$",
  "e-": r"e$^-$"
}
# Cases
cases = [4, 7, 9, 47, 62, 97]

In [3]:
# Time grid
tlim = [1e-14, 1e-7]
tdim = 200
t = np.geomspace(*tlim, num=tdim)

Load data and model

In [ ]:
data = utils.load_test_cases(path_to_data)
model = tf.keras.models.load_model(path_to_model)
init_sols = h5todict(path_to_init_sols)

Set variables names

In [ ]:
y_names = np.array(list(data.values())[0].columns)
y_names = np.concatenate([y_names[i] for i in indices["y"]]).tolist()
y_names = [name[2:] for name in y_names]
y_names

In [ ]:
y0_names = np.array(list(data.values())[0].columns)
y0_names = np.concatenate([y0_names[i] for i in indices["x"][:2]]).tolist()
y0_names = [name[2:-2] for name in y0_names]
y0_names

Plot true vs. predicted solutions

In [7]:
y0 = []
for (icase, idata) in data.items():
  idata = idata.to_numpy()
  case_nb = int(icase.split("_")[-1])
  if ((cases is not None) and (case_nb not in cases)):
    continue
  _, y_true, y_pred = utils.predict(model, idata, indices, t)
  y0i = [np.take(idata, i, axis=-1)[0] for i in indices["x"][:2]]
  y0i = np.concatenate(y0i).reshape(-1)
  y0.append(y0i)
  for yscale in ("linear", "log"):
    utils.plot_test(
      path=path_to_saving,
      x=t,
      y=y_true,
      y_pred=y_pred,
      varnames=[labels[name] for name in y_names],
      figname=icase+"_"+yscale,
      figext=["pdf", "pgf"],
      labels=[r"$t$ [s]", r"$n$ [m$^{-3}$]"],
      scales=["log", yscale],
      xlim=tlim,
      ylim=[1e12, np.amax(y_true)*2e0] if (yscale == "log") else None,
      save=True,
      show=False
    )

In [8]:
y0_df = pd.DataFrame(np.vstack(y0), columns=y0_names, index=cases)
y0_df.to_csv(path_to_saving+"/init_sols.csv", float_format="%.6e")